In [1]:
import os
from nemo_platform import NeMoPlatform

base_url = os.environ.get("NMP_BASE_URL", "http://localhost:8080")
sdk = NeMoPlatform(base_url=base_url, workspace="default")

In [6]:
import os
os.environ["NVIDIA_API_KEY"] = "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"

In [ ]:
import os
import data_designer.config as dd
from data_designer.interface import DataDesigner

model_configs_full = [
    dd.ModelConfig(
        alias="nvidia-text",
        provider="openai-provider",  # matches the name you just registered
        model="gpt-4o-mini",
    )
]
# Bump parallelism on the model config
model_configs_old = [
    dd.ModelConfig(
        alias="nvidia-text",
        provider="openai-provider",
        model="gpt-4o-mini",
        inference_parameters=dd.ChatCompletionInferenceParams(
            max_parallel_requests=20,  # up from default 4
            max_tokens=50,             # your outputs are just numbers/short strings
        ),
    )
]

model_configs = [
    dd.ModelConfig(
        alias="nvidia-text",
        provider="openai-provider",
        model="gpt-4o-mini",
        inference_parameters=dd.ChatCompletionInferenceParams(
            max_parallel_requests=3,  # stay under rate limit
            max_tokens=20,            # you only need a number/word
        ),
    )
]

config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
data_designer = DataDesigner()


In [8]:
import json
import importlib.util
import sys
from pathlib import Path

from utils.helpers import dedupe_chunks, chunks_to_context

from utils.rag import ManualRAG
from config import get_machine_config

cfg = get_machine_config("cnc_spindle")
rag = ManualRAG(cfg)

In [9]:
# ── 1. Schema queries: physical ranges & degradation-relevant parameters ──
schema_queries = [
    # Vibration degradation
    "vibration velocity severity classification ranges spindle operation",
    "vibration acceleration rms measurement spindle rpm thresholds",
    
    # Thermal degradation
    "motor temperature overheat detection level monitoring time constant",
    "motor overheat alarm threshold spindle thermal limit",
    
    # Mechanical wear signals
    "velocity error excess detection level spindle alarm",
    "position error spindle orientation pulses tolerance degradation",
    "synchronous error spindle pulses alarm threshold",
    
    # Load & torque degradation
    "torque command percentage maximum motor load detection signal",
    "load detection LDT1A motor output threshold spindle overload",
    "current overload alarm detection level motor current",
    
    # Speed/control degradation
    "speed arrival signal SARA detection range deviation",
    "actual motor speed versus commanded speed error",
    "motor voltage orientation servo mode degradation",
]

# ── 2. Process queries: behavioral/temporal degradation patterns ──
process_queries = [
    # Phase transition degradation
    "spindle orientation sequence states A B C D E timing degradation",
    "orientation time increase spindle wear completion delay",
    "acceleration deceleration time alarm detection disable threshold",

    # Fault progression
    "spindle alarm ALMA overspeed velocity error overload fault conditions",
    "alarm detection status excessive speed overload velocity error sequence",
    "torque limitation active fault condition trigger",

    # Degradation signatures
    "spindle motor speed command versus actual deviation wear indicator",
    "position error increase orientation completion failure",
    "vibration velocity rough classification spindle bearing wear",
    "motor current increase load torque degradation bearing friction",
]

# ── 3. Pull and dedupe ──
chunks_1 = rag.retrieve_chunks(schema_queries, n_results_per_query=4)
chunks_2 = rag.retrieve_chunks(process_queries, n_results_per_query=3)
chunks   = dedupe_chunks(chunks_1 + chunks_2)

In [ ]:
import os
import data_designer.config as dd
from data_designer.interface import DataDesigner

# ── Helper: filter chunks by keywords ──
def get_context(chunks, keywords, max_chunks=2, max_chars=800):
    """Pull only the most relevant chunks for a given column."""
    relevant = [
        c for c in chunks 
        if any(kw.lower() in c["text"].lower() for kw in keywords)
    ]
    text = "\n".join([c["text"] for c in relevant[:max_chunks]])
    return text[:max_chars]

# ── Pre-filter chunks by topic ──
ctx_speed      = get_context(chunks, ["motor speed", "spindle speed", "speed command", "SARA"])
ctx_velocity   = get_context(chunks, ["velocity error", "speed error", "deviation"])
ctx_position   = get_context(chunks, ["position error", "orientation", "pulses", "ORARA"])
ctx_torque     = get_context(chunks, ["torque command", "torque limitation", "maximum torque"])
ctx_vibration  = get_context(chunks, ["vibration velocity", "vibration acceleration", "mm/s", "severity"])
ctx_thermal    = get_context(chunks, ["temperature", "overheat", "thermal"])
ctx_current    = get_context(chunks, ["current", "overload", "motor current"])
ctx_alarm      = get_context(chunks, ["alarm", "ALMA", "fault", "overspeed", "overload detection"])
ctx_sequence   = get_context(chunks, ["orientation sequence", "state", "interval", "deceleration"])

# Print token estimates so you can sanity check
for name, ctx in [
    ("speed", ctx_speed), ("velocity", ctx_velocity), ("position", ctx_position),
    ("torque", ctx_torque), ("vibration", ctx_vibration), ("thermal", ctx_thermal),
    ("current", ctx_current), ("alarm", ctx_alarm), ("sequence", ctx_sequence),
]:
    print(f"{name:12s}: {len(ctx)} chars / ~{len(ctx)//4} tokens")

# ── Model config ──
model_configs = [
    dd.ModelConfig(
        alias="nvidia-text",
        provider="openai-provider",
        model="gpt-4o-mini",
        inference_parameters=dd.ChatCompletionInferenceParams(
            max_parallel_requests=3,
            max_tokens=20,
        ),
    )
]

# ── Base prompt — short, no RAG here ──
BASE_RULES = """You are a CNC spindle physics simulator. Output ONLY the requested value, no explanation.
Degradation rules:
- healthy:       nominal values, tight error bounds, low vibration
- early_wear:    slight increases in errors, vibration, temperature
- moderate_wear: noticeable deviations, rising torque, higher vibration
- severe_wear:   large errors, high vibration near alarm thresholds, possible faults
"""

data_designer = DataDesigner()
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

# ──────────────────────────────────────────────
# SAMPLER COLUMNS
# ──────────────────────────────────────────────

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="operation_mode",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["velocity_control", "orientation", "rigid_tapping", "synchronous_control"],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="degradation_level",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["healthy", "early_wear", "moderate_wear", "severe_wear"],
            weights=[0.5, 0.25, 0.15, 0.10],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="spindle_speed_command",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=500, high=12000),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="orientation_stop_position",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=0, high=4095),
    )
)

# ──────────────────────────────────────────────
# LLM COLUMNS — targeted RAG per column
# ──────────────────────────────────────────────

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="actual_spindle_speed",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_speed}

State: operation_mode={{{{ operation_mode }}}}, degradation={{{{ degradation_level }}}}, command={{{{ spindle_speed_command }}}} rpm
Output ONLY one decimal number (0-32767 rpm). Healthy: within 1% of command. Severe: up to 5% drift.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="velocity_error",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_velocity}

State: degradation={{{{ degradation_level }}}}, command={{{{ spindle_speed_command }}}} rpm, actual={{{{ actual_spindle_speed }}}} rpm
Output ONLY the error (actual minus command), a signed decimal in ±256 rpm. Healthy machines: between -5 and +5. Do NOT output the speed itself.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="position_error",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_position}

State: operation_mode={{{{ operation_mode }}}}, degradation={{{{ degradation_level }}}}, orientation_stop={{{{ orientation_stop_position }}}} pulses
Output ONLY one decimal number (range ±4096 pulses). Healthy: ±10. Severe: ±500 or more.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="torque_command",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_torque}

State: operation_mode={{{{ operation_mode }}}}, degradation={{{{ degradation_level }}}}, speed={{{{ spindle_speed_command }}}} rpm
Output ONLY one decimal number (0-100%). Healthy: 20-50%. Severe: can exceed 80%.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="vibration_velocity",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_vibration}

State: degradation={{{{ degradation_level }}}}, actual_speed={{{{ actual_spindle_speed }}}} rpm, torque={{{{ torque_command }}}}%
Output ONLY one decimal number (0-10.7 mm/s rms).
Scale: 0-1.8 healthy, 1.8-4.5 early, 4.5-7.1 moderate, 7.1-10.7 severe.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="vibration_acceleration",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_vibration}

State: degradation={{{{ degradation_level }}}}, vibration_velocity={{{{ vibration_velocity }}}} mm/s, speed={{{{ actual_spindle_speed }}}} rpm
Output ONLY one decimal number (0-0.4 g rms). Must correlate with vibration_velocity.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="motor_temperature",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_thermal}

State: degradation={{{{ degradation_level }}}}, torque={{{{ torque_command }}}}%, speed={{{{ actual_spindle_speed }}}} rpm
Output ONLY one decimal number (0-150°C). Healthy idle: 30-50. Heavy load: 60-80. Severe: 120-140.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="motor_current",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_current}

State: degradation={{{{ degradation_level }}}}, torque={{{{ torque_command }}}}%, temperature={{{{ motor_temperature }}}}°C
Output ONLY one decimal number (0-100 A). Rises with torque and bearing degradation.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="spindle_sequence_state",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_sequence}

State: operation_mode={{{{ operation_mode }}}}, degradation={{{{ degradation_level }}}}, position_error={{{{ position_error }}}} pulses
Output ONLY one letter: a, b, c, d, or e.
a=before orientation, b=speed reaching, c=linear decel, d=position loop closed, e=completed.
Severe wear: less likely to reach e.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="spindle_alarm",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_alarm}

State: degradation={{{{ degradation_level }}}}, velocity_error={{{{ velocity_error }}}} rpm, vibration={{{{ vibration_velocity }}}} mm/s, temp={{{{ motor_temperature }}}}°C, current={{{{ motor_current }}}} A
Output ONLY true or false.
Alarm if: velocity_error > 200, vibration > 9.0, temp > 130, current > 90.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="alarm_detection_status",
        model_alias="nvidia-text",
        prompt=BASE_RULES + f"""
Manual: {ctx_alarm}

State: spindle_alarm={{{{ spindle_alarm }}}}, torque={{{{ torque_command }}}}%, temp={{{{ motor_temperature }}}}°C
Output ONLY true or false. True if spindle_alarm is true OR secondary thresholds exceeded.
""",
    )
)

# ── Generate ──
results = data_designer.preview(config_builder)
results.display_sample_record()

[16:05:49] [INFO] 🔭 Preview generation in progress
[16:05:49] [INFO]   |-- 🔒 Jinja rendering engine: secure
[16:05:49] [INFO] ✅ Validation passed
[16:05:49] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[16:05:49] [INFO] 🩺 Running health checks for models...
[16:05:49] [INFO]   |-- 👀 Checking 'gpt-4o-mini' in provider named 'openai-provider' for model alias 'nvidia-text'...


speed       : 800 chars / ~200 tokens
velocity    : 800 chars / ~200 tokens
position    : 800 chars / ~200 tokens
torque      : 800 chars / ~200 tokens
vibration   : 800 chars / ~200 tokens
thermal     : 800 chars / ~200 tokens
current     : 800 chars / ~200 tokens
alarm       : 800 chars / ~200 tokens
sequence    : 800 chars / ~200 tokens


[16:05:49] [INFO]   |-- ✅ Passed!
[16:05:49] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue preview
[16:05:49] [INFO] 📝 llm-text model config for column 'actual_spindle_speed'
[16:05:49] [INFO]   |-- model: 'gpt-4o-mini'
[16:05:49] [INFO]   |-- model alias: 'nvidia-text'
[16:05:49] [INFO]   |-- model provider: 'openai-provider'
[16:05:49] [INFO]   |-- inference parameters:
[16:05:49] [INFO]   |  |-- generation_type=chat-completion
[16:05:49] [INFO]   |  |-- max_parallel_requests=3
[16:05:49] [INFO]   |  |-- max_tokens=20
[16:05:49] [INFO] 📝 llm-text model config for column 'position_error'
[16:05:49] [INFO]   |-- model: 'gpt-4o-mini'
[16:05:49] [INFO]   |-- model alias: 'nvidia-text'
[16:05:49] [INFO]   |-- model provider: 'openai-provider'
[16:05:49] [INFO]   |-- inference parameters:
[16:05:49] [INFO]   |  |-- generation_type=chat-completion
[16:05:49] [INFO]   |  |-- max_parallel_requests=3
[16:05:49] [INFO]   |  |-- max_tokens=20
[16:05:49] [INFO] 📝 llm-tex

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                                                        ┃ Value                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ operation_mode                                              │ synchronous_control                          │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ degradation_level                                           │ moderate_wear                                │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ spindle_speed_command                                       │ 5046.897612473863                            │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ orientation_stop_position                                   │ 970.9406305011394                            │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ actual_spindle_speed                                        │ 4806.9                                       │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ velocity_error                                              │ -240.0                                       │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ position_error                                              │ ±250.0                                       │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ torque_command                                              │ 73.2                                         │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ vibration_velocity                                          │ 6.4                                          │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ vibration_acceleration                                      │ 0.3                                          │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ motor_temperature                                           │ 86.5                                         │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ motor_current                                               │ 73.2                                         │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ spindle_sequence_state                                      │ c                                            │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ spindle_alarm                                               │ False                                        │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ alarm_detection_status                                      │ False                                        │
└─────────────────────────────────────────────────────────────┴──────────────────────────────────────────────┘

In [14]:
df = results.dataset
df

,operation_mode,degradation_level,spindle_speed_command,orientation_stop_position,position_error,actual_spindle_speed,spindle_sequence_state,velocity_error,torque_command,motor_temperature,motor_current,vibration_velocity,spindle_alarm,alarm_detection_status,vibration_acceleration
0,synchronous_control,moderate_wear,5046.897612,970.940631,±250.0,4806.9,c,-240.0,73.2,86.5,73.2,6.4,false,False,0.3
1,rigid_tapping,healthy,6550.988164,2372.844874,0.0,6550.9,e,6551.0,20.0,40.0,20.0,0.9,false,False,0.4
2,synchronous_control,early_wear,11091.975307,3009.465102,±35.2,11009.9,b,-81.1,60.5,68.0,60.5,4.2,false,False,0.3
3,orientation,early_wear,10425.666756,523.748566,±12.5,10412.0,b,-13.7,70.2,76.3,70.2,2.9,false,False,0.3
4,synchronous_control,healthy,1926.141806,2732.655622,0.0,1926.1,e,1926.1,30.0,39.0,30.0,0.5,false,False,0.4
5,synchronous_control,severe_wear,11829.108386,1650.403039,-490.6,11285.6,a,-200,86.3,135.0,86.3,10.6,true,True,0.4
6,synchronous_control,healthy,9099.299344,2329.847427,0.0,9099.3,e,9099.3,25.0,30.0,25.0,0.6,false,False,0.3
7,rigid_tapping,moderate_wear,578.741517,556.584494,±300.0,550.0,c,-28.7,65.0,90.0,65.0,6.3,false,false,0.3
8,velocity_control,healthy,2210.130623,1858.179472,-1.0,2210.1,e,2210.1,30.0,30.0,30.0,0.5,false,False,0.4
9,orientation,early_wear,4015.154533,3937.984194,±50.0,3971.0,c,-44.2,60.0,75.0,60.0,3.5,false,False,0.2


In [22]:
display(df)

,operation_mode,degradation_level,spindle_speed_command,orientation_stop_position,position_error,actual_spindle_speed,spindle_sequence_state,velocity_error,torque_command,motor_temperature,motor_current,vibration_velocity,spindle_alarm,alarm_detection_status,vibration_acceleration
0,synchronous_control,moderate_wear,5046.897612,970.940631,±250.0,4806.9,c,-240.0,73.2,86.5,73.2,6.4,false,False,0.3
1,rigid_tapping,healthy,6550.988164,2372.844874,0.0,6550.9,e,6551.0,20.0,40.0,20.0,0.9,false,False,0.4
2,synchronous_control,early_wear,11091.975307,3009.465102,±35.2,11009.9,b,-81.1,60.5,68.0,60.5,4.2,false,False,0.3
3,orientation,early_wear,10425.666756,523.748566,±12.5,10412.0,b,-13.7,70.2,76.3,70.2,2.9,false,False,0.3
4,synchronous_control,healthy,1926.141806,2732.655622,0.0,1926.1,e,1926.1,30.0,39.0,30.0,0.5,false,False,0.4
5,synchronous_control,severe_wear,11829.108386,1650.403039,-490.6,11285.6,a,-200,86.3,135.0,86.3,10.6,true,True,0.4
6,synchronous_control,healthy,9099.299344,2329.847427,0.0,9099.3,e,9099.3,25.0,30.0,25.0,0.6,false,False,0.3
7,rigid_tapping,moderate_wear,578.741517,556.584494,±300.0,550.0,c,-28.7,65.0,90.0,65.0,6.3,false,false,0.3
8,velocity_control,healthy,2210.130623,1858.179472,-1.0,2210.1,e,2210.1,30.0,30.0,30.0,0.5,false,False,0.4
9,orientation,early_wear,4015.154533,3937.984194,±50.0,3971.0,c,-44.2,60.0,75.0,60.0,3.5,false,False,0.2


In [15]:
print(df.shape)
print(df.columns.tolist())

(10, 15)
['operation_mode', 'degradation_level', 'spindle_speed_command', 'orientation_stop_position', 'position_error', 'actual_spindle_speed', 'spindle_sequence_state', 'velocity_error', 'torque_command', 'motor_temperature', 'motor_current', 'vibration_velocity', 'spindle_alarm', 'alarm_detection_status', 'vibration_acceleration']


In [18]:
# Basic stats on numeric columns
df.describe()

,spindle_speed_command,orientation_stop_position
count,10.000000,10.000000
mean,6277.410405,1994.265342
std,4134.279624,1106.703385
min,578.741517,523.748566
25%,2661.386601,1140.806233
50%,5798.942888,2094.013449
75%,10094.074903,2642.702935
max,11829.108386,3937.984194


In [19]:
# Check degradation distribution
df["degradation_level"].value_counts()

degradation_level
healthy          4
early_wear       3
moderate_wear    2
severe_wear      1
Name: count, dtype: int64

In [20]:
# Check alarm rates
df["spindle_alarm"].value_counts()

spindle_alarm
false    9
true     1
Name: count, dtype: int64

In [21]:
# Spot check physical consistency
df[["degradation_level", "vibration_velocity", "motor_temperature", 
          "torque_command", "spindle_alarm"]].sort_values("degradation_level")

,degradation_level,vibration_velocity,motor_temperature,torque_command,spindle_alarm
2,early_wear,4.2,68.0,60.5,false
3,early_wear,2.9,76.3,70.2,false
9,early_wear,3.5,75.0,60.0,false
1,healthy,0.9,40.0,20.0,false
4,healthy,0.5,39.0,30.0,false
6,healthy,0.6,30.0,25.0,false
8,healthy,0.5,30.0,30.0,false
0,moderate_wear,6.4,86.5,73.2,false
7,moderate_wear,6.3,90.0,65.0,false
5,severe_wear,10.6,135.0,86.3,true
